In [1]:
import os
import re
from itertools import chain
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import spacy
from spacy.tokens.doc import Doc
from spacy.tokens.span import Span
from spacy.tokens.token import Token
import coreferee
from coreferee.data_model import ChainHolder
from helpers import get_request
from bs4 import BeautifulSoup
from bs4.element import Tag

c:\Users\Adam\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\coreferee\manager.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("coreferee")

In [3]:
def get_text_from_file(path: str):
    text = ""
    
    try:
        with open(path, "r", encoding="utf8") as file:
            text = file.read()
    except Exception as e:
        print("Error reading file", path)
        print(e)
    finally:
        return text

In [4]:
articles_folder = "./articles"

In [5]:
article_files = os.listdir(articles_folder)
article_files = [file for file in article_files if file.endswith(".txt")]

In [6]:
files_subset = article_files[:5]

In [9]:
texts = [get_text_from_file(f"{articles_folder}/{file}") for file in article_files]

df = pd.DataFrame(data={
    "file": article_files,
    "text": texts
})
df

,file,text
0,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ..."
1,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...
2,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu..."
3,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...
4,news_10-most-memorable-ufc-moments-canada.txt,As we continue our Canadian-focused march thro...
...,...,...
724,news_zhang-mingyang-all-gas-no-brakes-ufc-kans...,Rising Chinese star Zhang Mingyang has a stat ...
725,news_zhang-mingyang-embracing-spotlight-ufc-sh...,Zhang Mingyang has all the makings of a potent...
726,news_zhang-mingyang-journey-main-event-light-h...,Zhang Mingyang first entered the UFC ecosystem...
727,news_zhang-weili-builds-her-case-goat-status.txt,


In [10]:
# This takes a long time, so have patience
df["doc"] = list(nlp.pipe(texts))

In [11]:
def get_people(doc: Doc):
    return [ent for ent in doc.ents if ent.label_ == "PERSON"]

In [12]:
df["people"] = df["doc"].apply(get_people)

In [13]:
all_people: list[Span] = list(chain.from_iterable(df["people"]))
all_names = set(p.text for p in all_people)
full_names = set(name for name in all_names 
                 if 
                 len(name.split()) >= 2 
                 and len(name.split()) <= 3
                 and re.search("^[a-zA-Z\s’]+$", name) is not None)

In [14]:
def is_woman_tag(tag: Tag):
    text = tag.get_text().strip().lower()
    return "woman" in text or "women" in text

def find_gender_on_ufc(name: str):
    name_query = "-".join(name.split()).replace('’', ' ')
    url = f"https://www.ufc.com/athlete/{name_query}"

    try:
        response = get_request(url)
        soup = BeautifulSoup(response.text, "html.parser")

        tags: list[Tag] = soup.find_all("p", "hero-profile__tag")
        if not tags:
            return None

        is_woman = any([is_woman_tag(tag) for tag in tags])
        if is_woman:
            return "woman"
        else:
            return "man"
    except Exception as e:
        print("Error getting gender of", name)
        print(e)

In [15]:
full_names

{'Malcolm Wellmaker',
 'Trey Ogden',
 'Rafael Cerquiera',
 'Ty Miller',
 'Javid Basharat',
 'Kevin Costner',
 'Steve Latrell',
 'Felipe Bunes',
 'Thiago Moises',
 'Lodune Sincaid',
 'Melissa Martinez',
 'Randa Markos',
 'An Tuan Ho',
 'Ryan Hall',
 'Henry Corrales',
 'Katlyn Cerminara',
 'Uncle Joe',
 'Akira Corassani',
 'Davide Scarano',
 'Gabe Green',
 'Viktoriia Dudakov',
 'Nazim Sadykhov',
 'Hyder Amil',
 'Ian Heinisch',
 'Germaine de Randamie',
 'Miguel Baeza',
 'Yanal Ashmouz',
 'Viktoriia Dudakva',
 'Czerwony Smok',
 'Timmy Cuambadef',
 'Caio Machado',
 'HanSeul Kim',
 'Justin Gaethje',
 'Bill Belichick',
 'Ricky Turcios',
 'Jason Parillo',
 'Gustavo Wurlitzer',
 'Michael Jordan',
 'Randy Brown',
 'Istela Nunes',
 'Carlos Leal',
 'Wilson Reis',
 'Eddie Wineland',
 'Sayif Saud',
 'Chael Sonnen',
 'Puro Chicali',
 'Leroy Duncan',
 'Oumar Sy',
 'Jayson Tatum',
 'Will Fiziev',
 'Magomed Gadhziyasulov',
 'nee Calderwood',
 'Salahdine Parnasse',
 'Will Chandler',
 'James Llontop',
 'J

In [16]:
# Set this to False if you want to fetch the genders from the UFC website
use_saved_gender_map = False
genders_map_file = "./genders_map.csv"

if use_saved_gender_map and os.path.exists(genders_map_file):
    name_gender_pairs_df = pd.read_csv(genders_map_file)
    genders_map: dict[str, str] = dict(zip(name_gender_pairs_df["name"], name_gender_pairs_df["gender"]))
else:
    def get_name_gender_pair_on_ufc(name: str):
        return (name, find_gender_on_ufc(name))

    with ThreadPoolExecutor() as executor:
        name_gender_pairs = [
            (n, g)
            for n, g in executor.map(get_name_gender_pair_on_ufc, full_names)
            if g is not None
        ]

    genders_map = dict(name_gender_pairs)

    for name, gender in name_gender_pairs:
        for name_component in name.split():
            genders_map[name_component] = gender

    pd.DataFrame(data={
        "name": genders_map.keys(),
        "gender": genders_map.values(),
    })\
        .set_index("name")\
        .to_csv(genders_map_file)

genders_map

{'Malcolm Wellmaker': 'man',
 'Trey Ogden': 'man',
 'Ty Miller': 'man',
 'Javid Basharat': 'man',
 'Felipe Bunes': 'man',
 'Thiago Moises': 'man',
 'Lodune Sincaid': 'man',
 'Melissa Martinez': 'woman',
 'Randa Markos': 'woman',
 'Ryan Hall': 'man',
 'Katlyn Cerminara': 'woman',
 'Akira Corassani': 'man',
 'Gabe Green': 'man',
 'Nazim Sadykhov': 'man',
 'Hyder Amil': 'man',
 'Ian Heinisch': 'man',
 'Germaine de Randamie': 'woman',
 'Miguel Baeza': 'man',
 'Yanal Ashmouz': 'man',
 'Caio Machado': 'man',
 'HanSeul Kim': 'man',
 'Justin Gaethje': 'man',
 'Ricky Turcios': 'man',
 'Randy Brown': 'man',
 'Istela Nunes': 'woman',
 'Carlos Leal': 'man',
 'Wilson Reis': 'man',
 'Eddie Wineland': 'man',
 'Oumar Sy': 'man',
 'James Llontop': 'man',
 'Joseph Lowry': 'man',
 'Eugene Jackson': 'man',
 'Aleksei Oleinik': 'man',
 'Maurice Greene': 'man',
 'Punahele Soriano': 'man',
 'Pete Sell': 'man',
 'Aleksandar Rakic': 'man',
 'Imanol Rodriguez': 'man',
 'SeokHyeon Ko': 'man',
 'Klidson Abreu': 'm

In [17]:
def get_coreference_genders(row: pd.Series):
    doc: Doc = row["doc"]

    chains: ChainHolder = doc._.coref_chains
    
    def loop():
        for chain in chains:
            main_item_mention = chain[chain.most_specific_mention_index]
            main_item = doc[main_item_mention.root_index]
            
            if main_item.text in genders_map:
                for mention in chain:
                    token = doc[mention.root_index]
                    yield (token, genders_map[main_item.text])
    
    return list(loop())

In [18]:
df["coreference_genders"] = df.apply(get_coreference_genders, axis="columns")

In [19]:
def get_people_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    people: list[Span] = row["people"]
    coreference_genders: list[tuple[Token, str]] = row["coreference_genders"]

    people_with_gender = [p for p in people if genders_map.get(p.text) == gender]
    coreferences_with_gender = [p for p, g in coreference_genders if g == gender]
    coreferences_with_gender = [Span(doc, p.i, p.i + 1) for p in coreferences_with_gender]
    
    return set(people_with_gender + coreferences_with_gender)

In [20]:
df["men"] = df.apply(get_people_with_gender, args=("man",), axis="columns")
df["women"] = df.apply(get_people_with_gender, args=("woman",), axis="columns")

In [21]:
def get_paragraph_boundaries(doc: Doc):
    line_breaks = [token for token in doc if token.text == "\n"]
    line_break_indexes = [lb.i for lb in line_breaks]
    paragraph_boundaries = list(zip([0] + line_break_indexes, line_break_indexes + [len(doc)]))
    return paragraph_boundaries

In [22]:
df["paragraph_boundaries"] = df["doc"].apply(get_paragraph_boundaries)

In [23]:
def count_genders_in_paragraphs(row: pd.Series, gender: str):
    people_with_gender: set[Span] = row[gender]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]

    def count_gender_in_paragraph(boundary: tuple[int, int]):
        start, end = boundary
        count = 0

        for person in people_with_gender:
            if person.end >= end:
                continue
            elif person.start < start:
                continue
            
            count += 1

        return count

    return [count_gender_in_paragraph(boundary) for boundary in paragraph_boundaries]

In [24]:
df["men_per_paragraph"] = df.apply(count_genders_in_paragraphs, args=("men",), axis="columns")
df["women_per_paragraph"] = df.apply(count_genders_in_paragraphs, args=("women",), axis="columns")

In [25]:
def assign_gender_to_paragraphs(row: pd.Series):
    men_per_paragraph: list[int] = row["men_per_paragraph"]
    women_per_paragraph: list[int] = row["women_per_paragraph"]

    def choose_gender(m: int, f: int):
        if m == 0 and f == 0:
            return None
        elif m > f:
            return "man"
        elif f > m:
            return "woman"
        else: 
            return "equal"

    return [choose_gender(m, f) for m, f in zip(men_per_paragraph, women_per_paragraph)]

In [26]:
df["paragraph_genders"] = df.apply(assign_gender_to_paragraphs, axis="columns")

In [27]:
def get_paragraphs_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]
    paragraph_genders: list[str | None] = row["paragraph_genders"]

    paragraphs_indexes_with_gender = [i for i, g in enumerate(paragraph_genders) if g == gender]
    paragraphs_boundaries_with_gender = [paragraph_boundaries[i] for i in paragraphs_indexes_with_gender]
    paragraphs_with_gender = [doc[start:end] for start, end in paragraphs_boundaries_with_gender]

    return paragraphs_with_gender

In [28]:
df["man_paragraphs"] = df.apply(get_paragraphs_with_gender, args=("man",), axis="columns")
df["woman_paragraphs"] = df.apply(get_paragraphs_with_gender, args=("woman",), axis="columns")
df["equal_paragraphs"] = df.apply(get_paragraphs_with_gender, args=("equal",), axis="columns")
df["genderless_paragraphs"] = df.apply(get_paragraphs_with_gender, args=(None,), axis="columns")

In [29]:
df

,file,text,doc,people,coreference_genders,men,women,paragraph_boundaries,men_per_paragraph,women_per_paragraph,paragraph_genders,man_paragraphs,woman_paragraphs,equal_paragraphs,genderless_paragraphs
0,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ...","(Buckle, up, ,, kids, ,, because, the, June, s...","[(Mario, Bautista), (Bautista), (Ricky, Simon)...","[(Bautista, man), (Bautista, man), (him, man),...","{(Kyoji, Horiguchi), (his), (Dvalishvili), (hi...","{(Erin, Blanchfield), (her), (Vieira), (Kayla,...","[(0, 14), (14, 94), (94, 164), (164, 174), (17...","[0, 0, 0, 0, 0, 2, 8, 3, 3, 0, 0, 6, 0, 1, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 9, 3, 0, ...","[None, None, None, None, None, man, man, man, ...","[(\n, A, couple, fights, before, the, bantamwe...","[(\n, Julianna, Peña, seeks, to, successfully,...",[],"[(Buckle, up, ,, kids, ,, because, the, June, ..."
1,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...,"(There, are, all, kinds, of, different, ways, ...","[(Bo, Nickal), (Gerald, Meerschaert), (Kevin, ...","[(Holland, man), (his, man), (he, man), (Nicka...","{(Sean, Brady), (He), (Zahabi), (his), (Michae...","{(Yan), (She), (Andrade), (his), (Shevchenko),...","[(0, 41), (41, 78), (78, 133), (133, 172), (17...","[0, 0, 0, 0, 0, 0, 1, 5, 2, 1, 4, 12, 9, 4, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 4, ...","[None, None, None, None, None, None, man, man,...","[(\n, While, both, Reinier, de, Ridder, and, B...","[(\n, Jessica, Andrade, and, Jasmine, Jasudavi...",[],"[(There, are, all, kinds, of, different, ways,..."
2,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu...","(On, Saturday, ,, February, 8, ,, the, UFC, re...","[(Alexander, Volkanovski), (Adesanya), (Dan, H...","[(Weili, woman), (her, woman), (Velasquez, man...","{(Johnson), (Hunt), (Chris, Clements), (Noguei...","{(Marion, Reneau), (Tatiana, Suarez), (she), (...","[(0, 30), (30, 67), (67, 159), (159, 220), (22...","[0, 0, 4, 2, 4, 7, 5, 7, 13, 9, 5, 0, 4, 4, 2,...","[0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, ...","[None, None, man, woman, man, man, man, man, m...","[(\n, Over, the, years, ,, the, stops, in, Syd...","[(\n, As, we, ready, to, see, Zhang, Weili, de...",[],"[(On, Saturday, ,, February, 8, ,, the, UFC, r..."
3,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...,"(Fourteen, individuals, have, held, the, UFC, ...","[(Dave, Menne), (Gil, Castillo), (Dricus, Du, ...","[(Franklin, man), (Franklin, man), (Quarry, ma...","{(Adesanya), (Quarry), (Adesanya), (Anderson, ...","{(Silva), (Silva), (Silva), (his), (he), (Silv...","[(0, 67), (67, 118), (118, 166), (166, 181), (...","[4, 0, 0, 3, 0, 4, 7, 2, 7, 3, 0, 3, 1, 7, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 3, 3, 2, ...","[man, None, None, man, None, man, man, man, ma...","[(Fourteen, individuals, have, held, the, UFC,...","[(\n, It, ’s, weird, to, try, to, think, back,...","[(\n, Anderson, Silva, defeats, Chael, Sonnen,...","[(\n, While, it, never, profiled, as, the, gla..."
4,news_10-most-memorable-ufc-moments-canada.txt,As we continue our Canadian-focused march thro...,"(As, we, continue, our, Canadian, -, focused, ...","[(Matt, Serra), (Serra), (Liddell), (Tito, Ort...","[(Serra, man), (Serra, man), (his, man), (He, ...","{(Liddell), (Franklin), (men), (Liddell), (Fra...","{(her), (Irene, Aldana), (Nina), (she), (Nunes...","[(0, 47), (47, 67), (67, 115), (115, 144), (14...","[0, 0, 0, 0, 0, 2, 0, 4, 7, 10, 5, 6, 2, 0, 0,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[None, None, None, None, None, man, None, man,...","[(\n, UFC, 83, marked, the, promotion, ’s, fir...","[(\n, When, I, spoke, with, Nunes, a, couple, ...",[],"[(As, we, continue, our, Canadian, -, focused,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
724,news_zhang-mingyang-all-gas-no-brakes-ufc-kans...,Rising Chin

In [ ]:
def save_separate_paragraphs(row: pd.Series):
    original_file: str = row["file"]
    new_folder = original_file.replace(".txt", "")

    os.makedirs(f"./articles/{new_folder}", exist_ok=True)

    man_paragraphs: list[Span] = row["man_paragraphs"]
    man_text = "\n".join([p.text for p in man_paragraphs])

    with open(f"./articles/{new_folder}/man.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(man_text)

    woman_paragraphs: list[Span] = row["woman_paragraphs"]
    woman_text = "\n".join([p.text for p in woman_paragraphs])

    with open(f"./articles/{new_folder}/woman.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(woman_text)

    equal_paragraphs: list[Span] = row["equal_paragraphs"]
    equal_text = "\n".join([p.text for p in equal_paragraphs])

    with open(f"./articles/{new_folder}/equal.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(equal_text)

    genderless_paragraphs: list[Span] = row["genderless_paragraphs"]
    genderless_text = "\n".join([p.text for p in genderless_paragraphs])

    with open(f"./articles/{new_folder}/genderless.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(genderless_text)

In [31]:
df.apply(save_separate_paragraphs, axis="columns")

0      None
1      None
2      None
3      None
4      None
       ... 
724    None
725    None
726    None
727    None
728    None
Length: 729, dtype: object